[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [4]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [6]:
import torch
import math

In [13]:
# ✏️ YOUR IMPLEMENTATION HERE

def create_mask(len_seq, window_size):
    mask = torch.zeros(len_seq, len_seq)
    for i in range(len_seq):
        left = max(i - window_size, 0)
        right = min(i+ window_size, len_seq - 1)
        mask[i][left:right+1] = 1
    return mask.bool()
    
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]

    batch, seq_len, d_k = K.shape

    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    mask = create_mask(seq_len, window_size)
    
    scores = scores.masked_fill(~mask, float('-inf'))

    attn = torch.softmax(scores, dim=-1)
    return torch.matmul(attn, V)    
    

In [11]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [12]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (0.6ms)
  ✅ [2/5] window_size=0 — only sees itself (0.3ms)
  ✅ [3/5] Large window equals full attention (5.9ms)
  ✅ [4/5] Distant tokens don't affect output (0.9ms)
  ✅ [5/5] Gradient flow (19.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (27.3ms total)
  Progress saved. Run status() to see your dashboard.

